In [82]:
# Import & Setup 
import pandas as pd
import numpy as np 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from pathlib import Path

FEATURES_DIR = Path().resolve().parent / 'data' / 'features'


print('Imports complete!')

Imports complete!


In [83]:
# Load & check data
df = pd.read_csv(FEATURES_DIR / 'model_dataset.csv')

print(df.head())

  Abbreviation  Year  RoundNumber  AvgFinishLast3  AvgFinishLast5  \
0          VER  2023            6           1.333             1.4   
1          ALO  2023            6           3.333             3.2   
2          OCO  2023            6          12.667            12.8   
3          HAM  2023            6           4.667             4.8   
4          RUS  2023            6          10.000             8.2   

   DNFRateLast5  AvgLapTime  AvgLapTimeLast3  AvgLapTimeLast5  AvgPitTime  \
0           0.0      82.814           93.127           94.138      24.484   
1           0.0      82.437           93.590           94.590     150.810   
2           0.4      83.467           94.433           95.535      25.312   
3           0.0      83.481           94.110           95.064      25.554   
4           0.2      83.460           94.707           95.365      30.875   

   ...  Corners  TrackLength  TotalLaps  TrackTemp  AirTemp  Rainfall  \
0  ...       19      2950.03       78.0      39.2

In [84]:
# Data preprocessing
# One-hot encode SafetyCar, VirtualSafetyCar, RedFlag
df[['SafetyCar', 'VirtualSafetyCar', 'RedFlag']] = df[['SafetyCar', 'VirtualSafetyCar', 'RedFlag']].astype(int)

# Drop Abbreviation, Year, RoundNumber & Event 
df = df.drop(columns=['Abbreviation', 'RoundNumber', 'Event'])

# Separate into features & target
X = df.loc[:, df.columns != 'Position']
y = df['Position']

print(f'X Features: \n{X}\n\n')
print(f'Target Feature: \n{y}\n')


X Features: 
     Year  AvgFinishLast3  AvgFinishLast5  DNFRateLast5  AvgLapTime  \
0    2023           1.333             1.4           0.0      82.814   
1    2023           3.333             3.2           0.0      82.437   
2    2023          12.667            12.8           0.4      83.467   
3    2023           4.667             4.8           0.0      83.481   
4    2023          10.000             8.2           0.2      83.460   
..    ...             ...             ...           ...         ...   
889  2025           3.333             5.8           0.0      89.932   
890  2025          12.667            12.8           0.2      89.648   
891  2025          10.667            12.2           0.2      90.188   
892  2025          13.000            14.6           0.0      89.681   
893  2025          14.667            15.4           0.0      89.849   

     AvgLapTimeLast3  AvgLapTimeLast5  AvgPitTime  AvgPitTimeLast3  \
0             93.127           94.138      24.484          419.9

In [85]:
# Create train & test split
X_train = X[(X['Year'] == 2023) | (X['Year'] == 2024)]
y_train = y[X_train.index]

X_test = X[(X['Year'] == 2025)]
y_test = y[X_test.index]

# Drop Year to prevent model skew
X_train = X_train.drop(columns=['Year'])
X_test = X_test.drop(columns=['Year'])

In [86]:
# Create Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

predictions = lr_model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R-squared Score: {r2}")

Mean Absolute Error: 2.9542138018369126
Mean Squared Error: 16.20008195876855
Root Mean Squared Error: 4.024932540896623
R-squared Score: 0.49699705099830715


In [99]:
# Create RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

rfr_model = RandomForestRegressor(random_state=42, 
                                  max_depth=50,
                                  n_estimators=10,
                                  min_samples_leaf=10,
                                  min_samples_split=5)
rfr_model.fit(X_train, y_train)

rfr_predictions = rfr_model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, rfr_predictions)
mse = mean_squared_error(y_test, rfr_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, rfr_predictions)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R-squared Score: {r2}")


Mean Absolute Error: 2.8897533595390854
Mean Squared Error: 15.275869903525937
Root Mean Squared Error: 3.908435736138684
R-squared Score: 0.5256932878737215


In [88]:
# Create XGBRegressor model
from xgboost import XGBRegressor

xgb_model = XGBRegressor(objective='reg:squarederror',
                         random_state=42,
                         n_estimators=200,
                         learning_rate=0.1,
                         max_depth=5)

xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, xgb_predictions)
mse = mean_squared_error(y_test, xgb_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, xgb_predictions)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R-squared Score: {r2}")

Mean Absolute Error: 2.9100580606834
Mean Squared Error: 16.21236764989807
Root Mean Squared Error: 4.026458450039944
R-squared Score: 0.4966155875659368
